# 🔬  AI Data Concierge - Reproducible Analysis

<a href="https://colab.research.google.com/" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

---

## 📋 Query
> **What is the most common complaint in downtown Pittsburgh last month?**

## 📊 Metadata
| Property | Value |
|----------|-------|
| **Generated** | 2026-10-07 15:44:44 |
| **Data Source** | WPRDC |
| **Sources Used** | Western PA Regional Data Center (WPRDC) |
| **Notebook Version** | 1.0 (Colab Compatible) |

---

## 📖 How to Use This Notebook

This notebook reproduces the exact analysis performed by the ** AI Data Concierge**.
Follow the steps below to verify, modify, or extend the analysis.

### ✅ Quick Start
1. **Run All Cells**: Click `Runtime` → `Run all` (or press `Ctrl+F9`)
2. **Wait for Setup**: The first cells install dependencies and configure the environment

### 🔧 What You Can Do
| Action | Description |
|--------|-------------|
| **Verify** | Run all cells to confirm the original results |
| **Modify** | Change parameters (dates, locations, filters) and re-run |
| **Extend** | Add your own analysis cells below the results |
| **Export** | Download results as CSV, or save notebook to Drive |

### 📚 Notebook Structure
1. **Setup** - Install dependencies (runs once in Colab)
2. **Configuration** - Import libraries and set up API connections
3. **Data Retrieval** - Fetch data from the data source
4. **Analysis** - Process and analyze the data
5. **Results** - View the final answer and confidence scores
6. **Citations** - Reference sources for your research

---


In [ ]:
# ============================================================
# STEP 1: Environment Setup
# ============================================================
# This cell installs all required packages for Google Colab.
# If running locally, you can skip this cell if packages are installed.

# Check if running in Google Colab
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🔧 Running in Google Colab - Installing dependencies...")
    !pip install -q pandas numpy matplotlib seaborn requests
    print("✅ Dependencies installed successfully!")
else:
    print("💻 Running locally - assuming dependencies are installed")
    print("   If not, run: pip install pandas numpy matplotlib seaborn requests")


In [ ]:
# ============================================================
# STEP 2: Import Libraries
# ============================================================
# These are the core libraries used throughout this notebook.
# Each import serves a specific purpose in the data pipeline.

import requests      # For making HTTP requests to data APIs
import pandas as pd  # For data manipulation and analysis
import numpy as np   # For numerical computations
from datetime import datetime  # For timestamp handling
import json          # For JSON parsing

# Visualization libraries (with graceful fallback)
try:
    import matplotlib.pyplot as plt  # For creating plots
    import seaborn as sns            # For statistical visualizations
    VISUALIZATION_AVAILABLE = True
    plt.style.use('seaborn-v0_8-whitegrid')
    print("📊 Visualization libraries loaded successfully")
except ImportError:
    VISUALIZATION_AVAILABLE = False
    print("⚠️ Visualization libraries not available")
    print("   Install with: pip install matplotlib seaborn")

# ============================================================
# STEP 3: Configuration
# ============================================================
# Data source configuration - modify these if needed

CKAN_URL = "https://data.wprdc.org"

# Display configuration info
print(f"\n📅 Notebook generated: {datetime.now().isoformat()}")
print(f"🔗 CKAN URL: {CKAN_URL}")
print(f"📌 Timestamp: 2026-10-07T15:44:44.728339")


In [ ]:
# ============================================================
# STEP 4: Helper Functions
# ============================================================
# These utility functions handle data fetching from the CKAN API.
# You can reuse these functions for your own data exploration.

def fetch_ckan_data(resource_id: str, limit: int = 10000, filters: dict = None) -> pd.DataFrame:
    """
    Fetch data from CKAN DataStore API.

    This function handles pagination automatically and returns all records
    up to the specified limit.

    Parameters:
    -----------
    resource_id : str
        The unique identifier for the CKAN resource (dataset)
    limit : int, default=10000
        Maximum number of records to fetch
    filters : dict, optional
        Field filters to apply (e.g., {"state": "CA"})

    Returns:
    --------
    pd.DataFrame
        A DataFrame containing the fetched records

    Example:
    --------
    >>> df = fetch_ckan_data("abc123", limit=1000, filters={"year": 2023})
    >>> print(f"Loaded {len(df)} records")
    """
    print(f"📥 Fetching data from resource: {resource_id}")

    all_records = []
    offset = 0
    batch_size = min(32000, limit)

    while offset < limit:
        params = {
            "resource_id": resource_id,
            "limit": min(batch_size, limit - offset),
            "offset": offset,
        }

        if filters:
            params["filters"] = filters

        response = requests.post(
            f"{CKAN_URL}/api/3/action/datastore_search",
            json=params,
            headers={"Content-Type": "application/json"},
        )

        if response.status_code != 200:
            print(f"❌ Error: {response.status_code} - {response.text}")
            break

        result = response.json()
        if not result.get("success"):
            print(f"❌ CKAN error: {result.get('error')}")
            break

        records = result.get("result", {}).get("records", [])
        if not records:
            break

        all_records.extend(records)
        offset += len(records)

        total = result.get("result", {}).get("total", 0)
        print(f"   Progress: {len(all_records):,} / {total:,} records")

        if offset >= total:
            break

    df = pd.DataFrame(all_records)

    # Remove CKAN internal fields that aren't useful for analysis
    internal_cols = ["_id", "_full_text"]
    df = df.drop(columns=[c for c in internal_cols if c in df.columns], errors="ignore")

    print(f"✅ Loaded {len(df):,} records with {len(df.columns)} columns")
    return df


def search_ckan_packages(query: str, rows: int = 10) -> list:
    """
    Search CKAN for datasets (packages) matching a query.

    Parameters:
    -----------
    query : str
        The search query (e.g., "employment statistics")
    rows : int, default=10
        Maximum number of results to return

    Returns:
    --------
    list
        A list of matching package dictionaries

    Example:
    --------
    >>> packages = search_ckan_packages("housing prices", rows=5)
    >>> for pkg in packages:
    ...     print(pkg['title'])
    """
    print(f"🔍 Searching for: '{query}'")

    response = requests.post(
        f"{CKAN_URL}/api/3/action/package_search",
        json={"q": query, "rows": rows},
        headers={"Content-Type": "application/json"},
    )

    if response.status_code != 200:
        print(f"❌ Search failed: {response.status_code}")
        return []

    result = response.json()
    if not result.get("success"):
        print(f"❌ Search error: {result.get('error')}")
        return []

    packages = result.get("result", {}).get("results", [])
    print(f"✅ Found {len(packages)} matching datasets")
    return packages


print("✅ Helper functions loaded successfully!")
print("   - fetch_ckan_data(resource_id, limit, filters)")
print("   - search_ckan_packages(query, rows)")


# Data Analysis Pipeline

The following steps show how the data was searched, loaded, and analyzed. Each step includes executable code you can modify and re-run.

---

## Step 1: Semantic Search Resources


**Result preview:**
```
Found 10 semantically matching resources for '311 complaints requests downtown Pittsburgh':

1. **Pittsburgh 311 Data** (49.3% match)
   Resource: 311 Data
   Resource ID: `5202679a-d243-402e-b82a-63189995a942`
   Dataset ID: `pittsburgh-311-data`
   Format: CSV | Rows: 934,488 | Cols: 26
   Tags: service_request, public_services, city_services, operations, infrastructure, issues, maintenance, reporting, compliance, utilities
   This dataset contains a comprehensive collection of case information, likely related to city services or public requests. The data spans a significant period, with cre
```


In [ ]:
# Step 1: Semantic Search Resources

# Resource discovery. The concierge used a semantic (vector) search
# over pre-indexed resource metadata here; the public equivalent is
# CKAN's keyword search, which reproduces the discovery step without
# private infrastructure.
import requests

params = {"q": '311 complaints requests downtown Pittsburgh', "rows": 10}
resp = requests.get("https://data.wprdc.org/api/3/action/package_search", params=params)
results = resp.json()["result"]["results"]

for ds in results:
    print(ds['title'])
    for r in ds.get("resources", []):
        print(f"  - {r['name']} ({r['format']}) ID: {r['id']}")


## Step 2: Semantic Search Resources


**Result preview:**
```
Found 10 semantically matching resources for 'police complaints or citizen requests by neighborhood':

1. **Police Incident Blotter (Archived)** (26.6% match)
   Resource: Historical Blotter Data
   Resource ID: `391942e2-25ef-43e4-8263-f8519fa8aada`
   Dataset ID: `uniform-crime-reporting-data`
   Format: CSV | Rows: 495,251 | Cols: 17
   Tags: crime_data, incident_reports, public_safety, location_based_data, geographic_analysis, statistical_analysis, criminal_activity, neighborhood_data, incident_type, time_series_data
   This dataset contains a comprehensive record of incidents, likely of a
```


In [ ]:
# Step 2: Semantic Search Resources

# Resource discovery. The concierge used a semantic (vector) search
# over pre-indexed resource metadata here; the public equivalent is
# CKAN's keyword search, which reproduces the discovery step without
# private infrastructure.
import requests

params = {"q": 'police complaints or citizen requests by neighborhood', "rows": 10}
resp = requests.get("https://data.wprdc.org/api/3/action/package_search", params=params)
results = resp.json()["result"]["results"]

for ds in results:
    print(ds['title'])
    for r in ds.get("resources", []):
        print(f"  - {r['name']} ({r['format']}) ID: {r['id']}")


## Step 3: Load Data from Resource

**Resource ID:** `5202679a-d243-402e-b82a-63189995a942`
**Limit:** 5

**Result preview:**
```
Resource: 5202679a-d243-402e-b82a-63189995a942
Total records: 974,591
Loaded: 5
Fields (25): case_number, status, case_owner, subject, subject_code, legacy_subject, created_date_et, created_date_utc, last_modified_date_et, last_modified_date_utc, closed_date_et, closed_date_utc, origin, street, street_id, city, neighborhood, census_tract, council_district, ward, police_zone, latitude, longitude, geo_accuracy, unique_id

Sample (5 rows):

case_number status     case_owner     subject subject_code                   legacy_subject         created_date_et        created_date_utc   last_modified_da
```


In [ ]:
# Step 3: Load Data from Resource

# Load resource data
import pandas as pd

params = {"resource_id": '5202679a-d243-402e-b82a-63189995a942', "limit": 5}
resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search", json=params)
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Total available: {result['total']:,}")
print(f"\nColumns: {', '.join(df.columns.tolist())}")
df.head(10)


## Step 4: SQL Analysis Query (exploratory attempt — failed)

This attempt errored during the analysis and was retried differently in a later step. It is preserved for provenance; its code is commented out so the notebook still runs end to end.

**SQL:**
```sql
SELECT DISTINCT "neighborhood" FROM "5202679a-d243-402e-b82a-63189995a942" WHERE "neighborhood" ILIKE '%down%' OR "neighborhood" ILIKE '%central business%'
```

**Result preview:**
```
Tool unavailable: run_sql_query is NOT available on this portal — its DataStore SQL endpoint returned HTTP 403, which means the endpoint is blocked or not enabled rather than the query being wrong. Do NOT call run_sql_query against this portal again; every retry will fail the same way. Use load_resource_data instead (it accepts filters, q, sort, fields, and limit) and aggregate the rows yourself. Portal said: ERROR: The request could not be satisfied 403 ERROR The request could not be satisfied. Request blocked. We can't connect to the server for this app or website at this time. There might b
```


In [ ]:
# Step 4: SQL Analysis Query — failed exploratory attempt, kept for provenance
# # Run SQL analysis query
# sql = 'SELECT DISTINCT "neighborhood" FROM "5202679a-d243-402e-b82a-63189995a942" WHERE "neighborhood" ILIKE \'%down%\' OR "neighborhood" ILIKE \'%central business%\''
# 
# resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
# result = resp.json()["result"]
# 
# df = pd.DataFrame(result["records"])
# df = df.drop(columns=["_id", "_full_text"], errors="ignore")
# 
# print(f"Query returned {len(df)} rows")
# df

## Step 5: SQL Analysis Query (exploratory attempt — failed)

This attempt errored during the analysis and was retried differently in a later step. It is preserved for provenance; its code is commented out so the notebook still runs end to end.

**SQL:**
```sql
SELECT MAX("created_date_et") as max_date, MIN("created_date_et") as min_date FROM "5202679a-d243-402e-b82a-63189995a942"
```

**Result preview:**
```
Tool unavailable: run_sql_query is NOT available on this portal — its DataStore SQL endpoint returned HTTP 403, which means the endpoint is blocked or not enabled rather than the query being wrong. Do NOT call run_sql_query against this portal again; every retry will fail the same way. Use load_resource_data instead (it accepts filters, q, sort, fields, and limit) and aggregate the rows yourself.
```


In [ ]:
# Step 5: SQL Analysis Query — failed exploratory attempt, kept for provenance
# # Run SQL analysis query
# sql = 'SELECT MAX("created_date_et") as max_date, MIN("created_date_et") as min_date FROM "5202679a-d243-402e-b82a-63189995a942"'
# 
# resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
# result = resp.json()["result"]
# 
# df = pd.DataFrame(result["records"])
# df = df.drop(columns=["_id", "_full_text"], errors="ignore")
# 
# print(f"Query returned {len(df)} rows")
# df

## Step 6: Load Data from Resource

**Resource ID:** `5202679a-d243-402e-b82a-63189995a942`
**Limit:** 5

**Result preview:**
```
Resource: 5202679a-d243-402e-b82a-63189995a942
Total records: 974,591
Loaded: 5
Fields (25): case_number, status, case_owner, subject, subject_code, legacy_subject, created_date_et, created_date_utc, last_modified_date_et, last_modified_date_utc, closed_date_et, closed_date_utc, origin, street, street_id, city, neighborhood, census_tract, council_district, ward, police_zone, latitude, longitude, geo_accuracy, unique_id

Sample (5 rows):

case_number status                 case_owner                              subject subject_code legacy_subject         created_date_et        created_date_utc
```


In [ ]:
# Step 6: Load Data from Resource

# Load resource data
import pandas as pd

params = {"resource_id": '5202679a-d243-402e-b82a-63189995a942', "limit": 5, "sort": 'created_date_et desc'}
resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search", json=params)
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Total available: {result['total']:,}")
print(f"\nColumns: {', '.join(df.columns.tolist())}")
df.head(10)


## Step 7: Load Data from Resource

**Resource ID:** `5202679a-d243-402e-b82a-63189995a942`
**Limit:** 5

**Result preview:**
```
Resource: 5202679a-d243-402e-b82a-63189995a942
Total records: 8
Loaded: 5
Fields (26): case_number, status, case_owner, subject, subject_code, legacy_subject, created_date_et, created_date_utc, last_modified_date_et, last_modified_date_utc, closed_date_et, closed_date_utc, origin, street, street_id, city, neighborhood, census_tract, council_district, ward, police_zone, latitude, longitude, geo_accuracy, unique_id, rank

Sample (5 rows):

case_number status             case_owner                                  subject  subject_code legacy_subject         created_date_et        created_date_ut
```


In [ ]:
# Step 7: Load Data from Resource

# Load resource data
import pandas as pd

params = {"resource_id": '5202679a-d243-402e-b82a-63189995a942', "limit": 5, "q": 'Downtown'}
resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search", json=params)
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Total available: {result['total']:,}")
print(f"\nColumns: {', '.join(df.columns.tolist())}")
df.head(10)


## Step 8: Load Data from Resource

**Resource ID:** `5202679a-d243-402e-b82a-63189995a942`
**Limit:** 500
**Filters:** `{"neighborhood": "Central Business District"}`

**Result preview:**
```
Resource: 5202679a-d243-402e-b82a-63189995a942
Total records: 24,424
Loaded: 500
Fields (4): case_number, subject, created_date_et, neighborhood

Sample (15 rows):

case_number                                 subject         created_date_et              neighborhood
26-00105453   City Facility or Building Maintenance 2026-10-06T17:16:40.000 Central Business District
26-00105407          Refuse or Recycling Violations 2026-10-06T16:00:20.000 Central Business District
26-00105338                                Potholes 2026-10-06T14:19:57.000 Central Business District
26-00105297    Abandoned Ve
```


In [ ]:
# Step 8: Load Data from Resource

# Load resource data
import pandas as pd

params = {"resource_id": '5202679a-d243-402e-b82a-63189995a942', "limit": 500, "filters": {"neighborhood": "Central Business District"}, "sort": 'created_date_et desc'}
resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search", json=params)
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Total available: {result['total']:,}")
print(f"\nColumns: {', '.join(df.columns.tolist())}")
df.head(10)


## Step 9: Load Data from Resource

**Resource ID:** `5202679a-d243-402e-b82a-63189995a942`
**Limit:** 50
**Filters:** `{"neighborhood": "Central Business District"}`

**Result preview:**
```
Resource: 5202679a-d243-402e-b82a-63189995a942
Total records: 24,424
Loaded: 50
Fields (3): case_number, subject, created_date_et

Sample (15 rows):

case_number                                 subject         created_date_et
26-00105453   City Facility or Building Maintenance 2026-10-06T17:16:40.000
26-00105407          Refuse or Recycling Violations 2026-10-06T16:00:20.000
26-00105338                                Potholes 2026-10-06T14:19:57.000
26-00105297    Abandoned Vehicle (Parked on Street) 2026-10-06T12:46:07.000
26-00105180 Narcotics ,Vice, or Gang Related Issues 2026-10-06T08:56:2
```


In [ ]:
# Step 9: Load Data from Resource

# Load resource data
import pandas as pd

params = {"resource_id": '5202679a-d243-402e-b82a-63189995a942', "limit": 50, "filters": {"neighborhood": "Central Business District"}, "sort": 'created_date_et desc'}
resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search", json=params)
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Total available: {result['total']:,}")
print(f"\nColumns: {', '.join(df.columns.tolist())}")
df.head(10)


## Step 10: Load Data from Resource

**Resource ID:** `5202679a-d243-402e-b82a-63189995a942`
**Limit:** 15
**Filters:** `{"neighborhood": "Central Business District"}`

**Result preview:**
```
Resource: 5202679a-d243-402e-b82a-63189995a942
Total records: 3,738
Loaded: 15
Fields (3): case_number, subject, created_date_et

Sample (15 rows):

case_number                                  subject         created_date_et
26-00103618 Road Work - Utility Cuts, Constructio... 2026-09-30T19:09:08.000
26-00103526         Catch Basins, Grates, and Sewers 2026-09-30T15:14:58.000
26-00103524         Catch Basins, Grates, and Sewers 2026-09-30T15:07:54.000
26-00103296 Traffic Signal or Pedestrian Signal R... 2026-09-30T08:38:55.000
26-00103143                      Water Runoff in ROW 2026-09-29T16
```


In [ ]:
# Step 10: Load Data from Resource

# Load resource data
import pandas as pd

params = {"resource_id": '5202679a-d243-402e-b82a-63189995a942', "limit": 15, "filters": {"neighborhood": "Central Business District"}, "q": '2026-09', "sort": 'created_date_et desc'}
resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search", json=params)
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Total available: {result['total']:,}")
print(f"\nColumns: {', '.join(df.columns.tolist())}")
df.head(10)


## Step 11: Load Data from Resource

**Resource ID:** `5202679a-d243-402e-b82a-63189995a942`
**Limit:** 15
**Filters:** `{"neighborhood": "Central Business District"}`

**Result preview:**
```
Resource: 5202679a-d243-402e-b82a-63189995a942
Total records: 24,424
Loaded: 15
Fields (3): case_number, subject, created_date_et

Sample (15 rows):

case_number                                  subject         created_date_et
26-00104785 Road Work - Utility Cuts, Constructio... 2026-10-05T09:08:16.000
26-00104748  Narcotics ,Vice, or Gang Related Issues 2026-10-05T08:24:30.000
26-00104585       Sidewalk/Curb/ADA Ramp Maintenance 2026-10-04T01:48:12.000
26-00104287                           Missed Pick Up 2026-10-02T10:49:47.000
26-00104196  Narcotics ,Vice, or Gang Related Issues 2026-10-02T0
```


In [ ]:
# Step 11: Load Data from Resource

# Load resource data
import pandas as pd

params = {"resource_id": '5202679a-d243-402e-b82a-63189995a942', "limit": 15, "filters": {"neighborhood": "Central Business District"}, "sort": 'created_date_et desc'}
resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search", json=params)
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Total available: {result['total']:,}")
print(f"\nColumns: {', '.join(df.columns.tolist())}")
df.head(10)


## Step 12: Load Data from Resource

**Resource ID:** `5202679a-d243-402e-b82a-63189995a942`
**Limit:** 30
**Filters:** `{"neighborhood": "Central Business District"}`

**Result preview:**
```
Resource: 5202679a-d243-402e-b82a-63189995a942
Total records: 24,424
Loaded: 30
Fields (3): case_number, subject, created_date_et

Sample (15 rows):

case_number                                 subject         created_date_et
26-00105453   City Facility or Building Maintenance 2026-10-06T17:16:40.000
26-00105407          Refuse or Recycling Violations 2026-10-06T16:00:20.000
26-00105338                                Potholes 2026-10-06T14:19:57.000
26-00105297    Abandoned Vehicle (Parked on Street) 2026-10-06T12:46:07.000
26-00105180 Narcotics ,Vice, or Gang Related Issues 2026-10-06T08:56:2
```


In [ ]:
# Step 12: Load Data from Resource

# Load resource data
import pandas as pd

params = {"resource_id": '5202679a-d243-402e-b82a-63189995a942', "limit": 30, "filters": {"neighborhood": "Central Business District"}, "sort": 'created_date_et desc'}
resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search", json=params)
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Total available: {result['total']:,}")
print(f"\nColumns: {', '.join(df.columns.tolist())}")
df.head(10)


## Step 13: Load Data from Resource

**Resource ID:** `5202679a-d243-402e-b82a-63189995a942`
**Limit:** 15
**Filters:** `{"neighborhood": "Central Business District"}`

**Result preview:**
```
Resource: 5202679a-d243-402e-b82a-63189995a942
Total records: 24,424
Loaded: 15
Fields (3): case_number, subject, created_date_et

Sample (15 rows):

case_number                                  subject         created_date_et
26-00103524         Catch Basins, Grates, and Sewers 2026-09-30T15:07:54.000
26-00103296 Traffic Signal or Pedestrian Signal R... 2026-09-30T08:38:55.000
26-00103143                      Water Runoff in ROW 2026-09-29T16:25:42.000
26-00102973      Water/Drinking Fountain Maintenance 2026-09-29T11:59:17.000
26-00102970                                Curb Cuts 2026-09-29T1
```


In [ ]:
# Step 13: Load Data from Resource

# Load resource data
import pandas as pd

params = {"resource_id": '5202679a-d243-402e-b82a-63189995a942', "limit": 15, "filters": {"neighborhood": "Central Business District"}, "sort": 'created_date_et desc'}
resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search", json=params)
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Total available: {result['total']:,}")
print(f"\nColumns: {', '.join(df.columns.tolist())}")
df.head(10)


## Step 14: Load Data from Resource

**Resource ID:** `5202679a-d243-402e-b82a-63189995a942`
**Limit:** 15
**Filters:** `{"neighborhood": "Central Business District"}`

**Result preview:**
```
Resource: 5202679a-d243-402e-b82a-63189995a942
Total records: 24,424
Loaded: 15
Fields (3): case_number, subject, created_date_et

Sample (15 rows):

case_number                              subject         created_date_et
26-00102159     Traffic and Speeding Enforcement 2026-09-26T18:58:54.000
26-00102106   Sidewalk/Curb/ADA Ramp Maintenance 2026-09-26T16:38:24.000
26-00101900   Sidewalk/Curb/ADA Ramp Maintenance 2026-09-25T22:09:45.000
26-00101859             Graffiti in Right of Way 2026-09-25T19:49:20.000
26-00101857                Graffiti on Structure 2026-09-25T19:35:06.000
26-00101856 
```


In [ ]:
# Step 14: Load Data from Resource

# Load resource data
import pandas as pd

params = {"resource_id": '5202679a-d243-402e-b82a-63189995a942', "limit": 15, "filters": {"neighborhood": "Central Business District"}, "sort": 'created_date_et desc'}
resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search", json=params)
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Total available: {result['total']:,}")
print(f"\nColumns: {', '.join(df.columns.tolist())}")
df.head(10)


## Step 15: Load Data from Resource

**Resource ID:** `5202679a-d243-402e-b82a-63189995a942`
**Limit:** 15
**Filters:** `{"neighborhood": "Central Business District"}`

**Result preview:**
```
Resource: 5202679a-d243-402e-b82a-63189995a942
Total records: 24,424
Loaded: 15
Fields (3): case_number, subject, created_date_et

Sample (15 rows):

case_number                                  subject         created_date_et
26-00101114                  Crashes and Close Calls 2026-09-24T00:00:50.000
26-00101076                          Illegal Parking 2026-09-23T17:34:29.000
26-00101027                               Overgrowth 2026-09-23T15:42:47.000
26-00101021                          Illegal Parking 2026-09-23T15:33:19.000
26-00101017                          Illegal Parking 2026-09-23T1
```


In [ ]:
# Step 15: Load Data from Resource

# Load resource data
import pandas as pd

params = {"resource_id": '5202679a-d243-402e-b82a-63189995a942', "limit": 15, "filters": {"neighborhood": "Central Business District"}, "sort": 'created_date_et desc'}
resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search", json=params)
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Total available: {result['total']:,}")
print(f"\nColumns: {', '.join(df.columns.tolist())}")
df.head(10)


## Step 16: Load Data from Resource

**Resource ID:** `5202679a-d243-402e-b82a-63189995a942`
**Limit:** 15
**Filters:** `{"neighborhood": "Central Business District"}`

**Result preview:**
```
Resource: 5202679a-d243-402e-b82a-63189995a942
Total records: 24,424
Loaded: 15
Fields (3): case_number, subject, created_date_et

Sample (15 rows):

case_number                                  subject         created_date_et
26-00100572                          Broken Sidewalk 2026-09-22T13:21:36.000
26-00100564                        Barricade Removal 2026-09-22T13:10:15.000
26-00100351              Building Maintenance Issues 2026-09-22T00:12:34.000
26-00100311                     Fire Safety Concerns 2026-09-21T18:11:00.000
26-00100265                           Missed Pick Up 2026-09-21T1
```


In [ ]:
# Step 16: Load Data from Resource

# Load resource data
import pandas as pd

params = {"resource_id": '5202679a-d243-402e-b82a-63189995a942', "limit": 15, "filters": {"neighborhood": "Central Business District"}, "sort": 'created_date_et desc'}
resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search", json=params)
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Total available: {result['total']:,}")
print(f"\nColumns: {', '.join(df.columns.tolist())}")
df.head(10)


## Step 17: Load Data from Resource

**Resource ID:** `5202679a-d243-402e-b82a-63189995a942`
**Limit:** 15
**Filters:** `{"neighborhood": "Central Business District"}`

**Result preview:**
```
Resource: 5202679a-d243-402e-b82a-63189995a942
Total records: 24,424
Loaded: 15
Fields (3): case_number, subject, created_date_et

Sample (15 rows):

case_number                                  subject         created_date_et
26-00099864     Work or Construction Without Permits 2026-09-18T15:12:36.000
26-00099856 Pedestrian Safety Concerns - Non Sign... 2026-09-18T14:32:32.000
26-00099853                    Street Light Concerns 2026-09-18T14:29:34.000
26-00099852                    Street Light Concerns 2026-09-18T14:28:26.000
26-00099838                          Illegal Dumping 2026-09-18T1
```


In [ ]:
# Step 17: Load Data from Resource

# Load resource data
import pandas as pd

params = {"resource_id": '5202679a-d243-402e-b82a-63189995a942', "limit": 15, "filters": {"neighborhood": "Central Business District"}, "sort": 'created_date_et desc'}
resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search", json=params)
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Total available: {result['total']:,}")
print(f"\nColumns: {', '.join(df.columns.tolist())}")
df.head(10)


## Step 18: Load Data from Resource

**Resource ID:** `5202679a-d243-402e-b82a-63189995a942`
**Limit:** 15
**Filters:** `{"neighborhood": "Central Business District"}`

**Result preview:**
```
Resource: 5202679a-d243-402e-b82a-63189995a942
Total records: 24,424
Loaded: 15
Fields (3): case_number, subject, created_date_et

Sample (15 rows):

case_number                                  subject         created_date_et
26-00099127      Bicycle Traffic and Safety Concerns 2026-09-16T17:43:24.000
26-00098862                    Street Light Concerns 2026-09-16T11:09:11.000
26-00098837                                 Homeless 2026-09-16T10:32:01.000
26-00098818                    Graffiti on Structure 2026-09-16T10:03:26.000
26-00098688                    Street Light Concerns 2026-09-15T2
```


In [ ]:
# Step 18: Load Data from Resource

# Load resource data
import pandas as pd

params = {"resource_id": '5202679a-d243-402e-b82a-63189995a942', "limit": 15, "filters": {"neighborhood": "Central Business District"}, "sort": 'created_date_et desc'}
resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search", json=params)
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Total available: {result['total']:,}")
print(f"\nColumns: {', '.join(df.columns.tolist())}")
df.head(10)


## Step 19: Load Data from Resource

**Resource ID:** `5202679a-d243-402e-b82a-63189995a942`
**Limit:** 15
**Filters:** `{"neighborhood": "Central Business District"}`

**Result preview:**
```
Resource: 5202679a-d243-402e-b82a-63189995a942
Total records: 24,424
Loaded: 15
Fields (3): case_number, subject, created_date_et

Sample (15 rows):

case_number                              subject         created_date_et
26-00098228                             Homeless 2026-09-15T08:18:54.000
26-00098217            Litter on Public Property 2026-09-15T08:04:42.000
26-00098186                Street Light Concerns 2026-09-14T22:05:23.000
26-00098008            Paving Request or Concern 2026-09-14T13:48:54.000
26-00097953   Special Events Question or Concern 2026-09-14T12:48:05.000
26-00097824 
```


In [ ]:
# Step 19: Load Data from Resource

# Load resource data
import pandas as pd

params = {"resource_id": '5202679a-d243-402e-b82a-63189995a942', "limit": 15, "filters": {"neighborhood": "Central Business District"}, "sort": 'created_date_et desc'}
resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search", json=params)
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Total available: {result['total']:,}")
print(f"\nColumns: {', '.join(df.columns.tolist())}")
df.head(10)


## Step 20: Load Data from Resource

**Resource ID:** `5202679a-d243-402e-b82a-63189995a942`
**Limit:** 15
**Filters:** `{"neighborhood": "Central Business District"}`

**Result preview:**
```
Resource: 5202679a-d243-402e-b82a-63189995a942
Total records: 24,424
Loaded: 15
Fields (3): case_number, subject, created_date_et

Sample (15 rows):

case_number                          subject         created_date_et
26-00097275 Catch Basins, Grates, and Sewers 2026-09-11T18:03:14.000
26-00097149                      Dead Animal 2026-09-11T15:41:02.000
26-00097122                   Missed Pick Up 2026-09-11T15:12:49.000
26-00096858         Repair or Replace a Sign 2026-09-11T10:49:14.000
26-00096817                         Potholes 2026-09-11T09:57:58.000
26-00096537            Police - Subm
```


In [ ]:
# Step 20: Load Data from Resource

# Load resource data
import pandas as pd

params = {"resource_id": '5202679a-d243-402e-b82a-63189995a942', "limit": 15, "filters": {"neighborhood": "Central Business District"}, "sort": 'created_date_et desc'}
resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search", json=params)
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Total available: {result['total']:,}")
print(f"\nColumns: {', '.join(df.columns.tolist())}")
df.head(10)


## Step 21: Load Data from Resource

**Resource ID:** `5202679a-d243-402e-b82a-63189995a942`
**Limit:** 15
**Filters:** `{"neighborhood": "Central Business District"}`

**Result preview:**
```
Resource: 5202679a-d243-402e-b82a-63189995a942
Total records: 24,424
Loaded: 15
Fields (3): case_number, subject, created_date_et

Sample (15 rows):

case_number                                  subject         created_date_et
26-00095582                                 Potholes 2026-09-09T09:16:07.000
26-00095532                          Illegal Parking 2026-09-09T08:13:07.000
26-00095521              Building Maintenance Issues 2026-09-09T07:56:15.000
26-00095396           Refuse or Recycling Violations 2026-09-08T15:49:38.000
26-00095327                Litter on Public Property 2026-09-08T1
```


In [ ]:
# Step 21: Load Data from Resource

# Load resource data
import pandas as pd

params = {"resource_id": '5202679a-d243-402e-b82a-63189995a942', "limit": 15, "filters": {"neighborhood": "Central Business District"}, "sort": 'created_date_et desc'}
resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search", json=params)
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Total available: {result['total']:,}")
print(f"\nColumns: {', '.join(df.columns.tolist())}")
df.head(10)


## Step 22: Load Data from Resource

**Resource ID:** `5202679a-d243-402e-b82a-63189995a942`
**Limit:** 15
**Filters:** `{"neighborhood": "Central Business District"}`

**Result preview:**
```
Resource: 5202679a-d243-402e-b82a-63189995a942
Total records: 24,424
Loaded: 15
Fields (3): case_number, subject, created_date_et

Sample (15 rows):

case_number                               subject         created_date_et
26-00094524           Building Maintenance Issues 2026-09-04T15:03:36.000
26-00094185            Construction Site Concerns 2026-09-04T09:03:18.000
26-00094149                       Police Concerns 2026-09-04T08:29:08.000
26-00093973  Traffic and Pedestrian Signal Repair 2026-09-03T15:47:04.000
26-00093869      Catch Basins, Grates, and Sewers 2026-09-03T12:33:22.000
26-000
```


In [ ]:
# Step 22: Load Data from Resource

# Load resource data
import pandas as pd

params = {"resource_id": '5202679a-d243-402e-b82a-63189995a942', "limit": 15, "filters": {"neighborhood": "Central Business District"}, "sort": 'created_date_et desc'}
resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search", json=params)
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Total available: {result['total']:,}")
print(f"\nColumns: {', '.join(df.columns.tolist())}")
df.head(10)


## Step 23: Load Data from Resource

**Resource ID:** `5202679a-d243-402e-b82a-63189995a942`
**Limit:** 15
**Filters:** `{"neighborhood": "Central Business District"}`

**Result preview:**
```
Resource: 5202679a-d243-402e-b82a-63189995a942
Total records: 24,424
Loaded: 15
Fields (3): case_number, subject, created_date_et

Sample (15 rows):

case_number                                  subject         created_date_et
26-00093206                          Police Concerns 2026-09-02T08:11:46.000
26-00093197                Litter on Public Property 2026-09-02T08:03:55.000
26-00093127                          Police Concerns 2026-09-01T17:55:59.000
26-00092911                 Repair or Replace a Sign 2026-09-01T11:30:36.000
26-00092830 Traffic Signal or Pedestrian Signal R... 2026-09-01T0
```


In [ ]:
# Step 23: Load Data from Resource

# Load resource data
import pandas as pd

params = {"resource_id": '5202679a-d243-402e-b82a-63189995a942', "limit": 15, "filters": {"neighborhood": "Central Business District"}, "sort": 'created_date_et desc'}
resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search", json=params)
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Total available: {result['total']:,}")
print(f"\nColumns: {', '.join(df.columns.tolist())}")
df.head(10)


## Step 24: Load Data from Resource

**Resource ID:** `5202679a-d243-402e-b82a-63189995a942`
**Limit:** 15
**Filters:** `{"neighborhood": "Central Business District"}`

**Result preview:**
```
Resource: 5202679a-d243-402e-b82a-63189995a942
Total records: 24,424
Loaded: 15
Fields (3): case_number, subject, created_date_et

Sample (15 rows):

case_number                                 subject         created_date_et
26-00092006        Traffic and Speeding Enforcement 2026-08-30T12:32:53.000
26-00091960                         Illegal Parking 2026-08-30T09:21:16.000
26-00091959                    Fire Safety Concerns 2026-08-30T09:05:16.000
26-00091944                         Traffic Calming 2026-08-29T22:40:07.000
26-00091931          Refuse or Recycling Violations 2026-08-29T18:56:2
```


In [ ]:
# Step 24: Load Data from Resource

# Load resource data
import pandas as pd

params = {"resource_id": '5202679a-d243-402e-b82a-63189995a942', "limit": 15, "filters": {"neighborhood": "Central Business District"}, "sort": 'created_date_et desc'}
resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search", json=params)
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Total available: {result['total']:,}")
print(f"\nColumns: {', '.join(df.columns.tolist())}")
df.head(10)


# ============================================================
# 📊 RESULTS
# ============================================================

## 💡 Answer (as delivered in chat)

> **I wasn't able to complete the analysis for this question. It may help to rephrase it — for example:

- Name a specific place (a city, county, or state)
- Name a specific time period (a year or range of years)
- Ask about one metric at a time

You can also ask *"What datasets are available on this topic?"* to see what I can search.**

*The figures above restate the assistant's answer for convenience — they are
not computed by this cell. The derivation lives in the code cells earlier in
this notebook; re-run them to re-derive every number from the source APIs.*

---

## 🎯 Confidence Assessment

The Data Concierge evaluates the reliability of its answer using multiple factors:


> ⚠️ Confidence score not available for this query.


# ============================================================
# 📐 CONFIDENCE SCORING METHODOLOGY
# ============================================================

## How We Calculate Confidence

The Data Concierge uses a **weighted composite score** to assess the reliability
of each answer. The final confidence score is a weighted average of five independent
factors, each measuring a different aspect of answer quality.

### Scoring Formula

```
Final Score = (0.25 × Query Interpretation)
            + (0.25 × Source Authority)
            + (0.20 × Retrieval Match)
            + (0.15 × Data Recency)
            + (0.15 × Computation Reliability)
```

### Factor Descriptions

| Factor | Weight | What It Measures | How It's Calculated |
|--------|--------|------------------|---------------------|
| **Query Interpretation** | 25% | How well the system understood the query | Entity extraction confidence × intent classification confidence |
| **Source Authority** | 25% | Trustworthiness of the data source | Pre-assigned per source (BLS/Census: 0.95, Data Commons: 0.90, CKAN: 0.85) |
| **Retrieval Match** | 20% | How well the retrieved data matches the query | Retrieval score, boosted by observation count (up to 5 observations) |
| **Data Recency** | 15% | How fresh the data is | 1.0 if within expected update cycle, decays to 0.4 floor for older data |
| **Computation Reliability** | 15% | Accuracy of the computation method | By type: direct lookup 1.0, trend analysis 0.85, statistical inference 0.70 |

### Confidence Levels

| Level | Score Range | Interpretation |
|-------|-------------|----------------|
| 🟢 **HIGH** | ≥ 85% | Results are reliable and well-supported by authoritative data |
| 🟡 **MEDIUM** | 50% – 84% | Results are reasonable but may benefit from verification |
| 🔴 **LOW** | 25% – 49% | Results should be treated with caution; data may be incomplete |
| ⚫ **VERY LOW** | < 25% | Insufficient data; consider alternative sources or queries |

### Source Authority Ratings

| Data Source | Authority Score | Rationale |
|-------------|----------------|-----------|
| Bureau of Labor Statistics (BLS) | 0.95 | Official federal statistics, rigorous methodology |
| U.S. Census Bureau | 0.95 | Comprehensive national data collection |
| Bureau of Economic Analysis (BEA) | 0.95 | Official GDP and economic accounts |
| FRED (Federal Reserve) | 0.95 | Curated economic data from the Fed |
| Google Data Commons | 0.90 | Aggregated from authoritative sources |
| WPRDC (Pittsburgh) | 0.88 | Curated regional open data portal |
| Generic CKAN Portals | 0.85 | Quality varies by portal and dataset |

### Data Recency Decay

The recency score decays based on how old the data is relative to its expected
update frequency:

- **Within 1× update cycle**: 1.0 (fully current)
- **Within 2× update cycle**: 0.8
- **Within 4× update cycle**: 0.6
- **Older than 4× update cycle**: 0.4 (floor)

### Escalation Policy

When the final confidence score falls **below 50%** after **2 retrieval attempts**,
the system flags the query for human review rather than providing a potentially
unreliable answer.

---


# ============================================================
# 📚 CITATIONS & REFERENCES
# ============================================================

## 📖 Data Sources

## Data Sources and Citations

**[1]** Western PA Regional Data Center (WPRDC)
- Dataset: Western PA Regional Data Center (WPRDC)
- URL: [https://data.wprdc.org](https://data.wprdc.org)
- Accessed: 2026-10-07

---

## 🔄 Reproducibility Guide

This notebook was automatically generated by the ** AI Data Concierge**.
Follow these steps to reproduce or extend the analysis:

### Prerequisites

```bash
pip install pandas numpy requests matplotlib seaborn
```

### Running the Notebook

| Step | Action | Notes |
|------|--------|-------|
| 1 | **Open in Colab** | Click the "Open in Colab" badge at the top |
| 2 | **Run All Cells** | `Runtime` → `Run all` or `Ctrl+F9` |
| 3 | **Wait for completion** | Dependencies install automatically in Colab |
| 4 | **Review results** | Scroll down to see the analysis results |

### ⚠️ Important Notes

- **Data freshness**: Results may differ if data sources have been updated since generation
- **API limits**: Some data sources have rate limits; wait if you encounter errors
- **Modifications**: Feel free to modify parameters and re-run cells to explore further

### 📅 Generation Info

- **Generated**: 2026-10-07 15:44:44
- **Query**: What is the most common complaint in downtown Pittsburgh last month?
- **Data Source**: WPRDC

---

*Generated by  AI Data Concierge v0.1.0*
